# Combining Different Models: Voting and Stacking

So far we've seen two ensemble strategies that combine many copies of the *same* model type. Bagging (Random Forests) trains independent trees on bootstrap samples and averages them - reducing variance. Boosting (Gradient Boosting, XGBoost) trains shallow trees sequentially, each correcting the last - reducing bias.

Both are powerful, but they're limited to one model family. What if a linear model captures one part of the pattern well, a tree captures another, and an SVM captures a third? Voting and stacking let you combine *different* model types into a single ensemble, drawing on each model's strengths.

This lecture also introduces nested cross-validation - the proper way to evaluate models that involve hyperparameter tuning. Without it, the CV scores you've been reporting are optimistically biased. We'll see exactly how much.

## Setup

In [ ]:
%matplotlib inline

import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier,
    StackingClassifier,
    VotingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [ ]:
# Synthetic dataset: 20 features, 8 informative, 4 redundant, 8 noise
# - class_sep=0.8 and flip_y=0.12 make the problem hard enough that
#   no single model dominates
# - n_clusters_per_class=2 creates non-convex class regions that
#   different model types handle differently
X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=8,
    n_redundant=4,
    n_repeated=0,
    n_classes=2,
    n_clusters_per_class=2,
    class_sep=0.8,
    flip_y=0.12,
    random_state=42,
)

print(f"Dataset: {X.shape[0]:,} samples, {X.shape[1]} features")
print(f"Class balance: {y.mean():.1%} positive")

With 20 features we can't visualize the full dataset directly. PCA (Principal Component Analysis) helps by finding the directions of maximum variance in the data and projecting onto them. Think of it as finding the best 2D "shadow" of a 20-dimensional object - it compresses the data while preserving as much of the spread as possible. We'll cover PCA properly in 7130; for now it's just a visualization tool.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

X_2d = PCA(n_components=2, random_state=42).fit_transform(X)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: colored by class label
scatter = ax1.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap="RdYlBu", alpha=0.4, s=12, edgecolors="none")
ax1.set_xlabel("PC1")
ax1.set_ylabel("PC2")
ax1.set_title("Colored by Class Label")
ax1.legend(*scatter.legend_elements(), title="Class")

# Right: colored by cluster (GMM finds the hidden structure)
gmm = GaussianMixture(n_components=4, random_state=42).fit(X_2d)
clusters = gmm.predict(X_2d)
cluster_colors = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3"]  # saturated, distinct
for k in range(4):
    mask = clusters == k
    ax2.scatter(X_2d[mask, 0], X_2d[mask, 1], c=cluster_colors[k], alpha=0.5, s=12,
                edgecolors="none", label=f"Cluster {k}")
ax2.set_xlabel("PC1")
ax2.set_ylabel("PC2")
ax2.set_title("Colored by Cluster (GMM, k=4)")
ax2.legend(title="Cluster")

plt.tight_layout()
plt.show()

The left panel shows the data colored by class label - it looks like a noisy mess with no clean boundary. The right panel reveals the hidden structure using a *clustering* algorithm (Gaussian Mixture Model). Clustering groups data points by similarity without using labels - it's looking purely at the geometry of the feature space. Here it finds 4 distinct groups. Since we built this dataset with `n_clusters_per_class=2`, that's exactly right: each class is made of 2 clusters, and those clusters interleave.

This explains why LogReg struggles - it can only draw a single linear boundary through this, which can't separate interleaved clusters. Trees can carve out irregular regions with axis-aligned splits, and SVM with an RBF kernel can wrap flexible boundaries around each cluster. Different tools for different structures.

Clustering and other methods for finding patterns in data *without* labels - dimensionality reduction, association rules, anomaly detection - are the focus of INSY 7130. Here we have labels, so the question is which supervised models best exploit the structure. Different models handle it differently, which is exactly why combining them might help.

---

## Part 1: Individual Baselines

Before combining models, we need to see how they perform individually. Each model type has different strengths on this dataset - the redundant and noisy features will affect them differently. Linear models are sensitive to redundancy, KNN is sensitive to irrelevant features, and trees ignore both but may overfit.

In [ ]:
# Individual models to compare
individual_models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "LogReg": Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(random_state=42))]),
    "KNN": Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier())]),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, random_state=42),
    "SVC": Pipeline([("scaler", StandardScaler()), ("svc", SVC(random_state=42))]),
}

baseline_rows = []
for name, model in individual_models.items():
    cv = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    baseline_rows.append({
        "Model": name,
        "CV Accuracy (mean)": f"{cv.mean():.4f}",
        "CV Accuracy (std)": f"{cv.std():.4f}",
    })

baseline_df = pd.DataFrame(baseline_rows)
print(baseline_df.to_string(index=False))

The models disagree - that's what we want. If every model scored the same, combining them would add nothing. The *gaps* between models suggest that each is capturing different aspects of the data structure. Let's see if we can exploit those differences.

---

## Part 2: Voting Ensembles

### Combining predictions by vote

The simplest way to combine different models is to let them vote. For classification, there are two flavors:

- *Hard voting*: each model predicts a class, the ensemble takes the majority. Simple democracy.
- *Soft voting*: each model predicts *probabilities*, the ensemble averages them, then picks the class with the highest average probability. More nuanced because it weights confident predictions more heavily.

Soft voting generally outperforms hard voting when the individual models produce well-calibrated probabilities. We'll use soft voting here.

In [ ]:
# Three diverse models: linear, tree-based, kernel-based
log_clf = Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(random_state=42))])
rf_clf = RandomForestClassifier(n_estimators=200, random_state=42)
svc_clf = Pipeline([("scaler", StandardScaler()), ("svc", SVC(probability=True, random_state=42))])

voting_v1 = VotingClassifier(
    estimators=[("lr", log_clf), ("rf", rf_clf), ("svc", svc_clf)],
    voting="soft",
)

# Compare with individual models
models_to_compare = {
    "LogReg": log_clf,
    "Random Forest": rf_clf,
    "SVC": svc_clf,
    "Voting (LR+RF+SVC)": voting_v1,
}

for name, model in models_to_compare.items():
    cv = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    print(f"{name:<25s}  CV Accuracy: {cv.mean():.4f} ± {cv.std():.4f}")

Voting may not beat every individual model. If one member is substantially weaker than the others, it drags the average down. LogReg struggles on this dataset (non-convex boundaries, noisy features), and soft voting gives it equal weight alongside RF and SVC. The ensemble is only as strong as its weakest member.

What if we drop the weak member?

In [ ]:
# Voting without the weak member
voting_v2 = VotingClassifier(
    estimators=[("rf", rf_clf), ("svc", svc_clf)],
    voting="soft",
)

cv_v2 = cross_val_score(voting_v2, X, y, cv=5, scoring="accuracy")
print(f"Voting (RF+SVC only):   CV Accuracy: {cv_v2.mean():.4f} ± {cv_v2.std():.4f}")

Removing the weak member improves the ensemble. This is the key lesson: voting works best when all members are reasonably strong. A weak model adds noise, not signal.

We'll use the RF+SVC voting ensemble for the final comparison.

In [ ]:
voting_clf = voting_v2  # use the better voting ensemble going forward

### Why diversity matters

An ensemble is only as good as the diversity of its members. If all three models make the same predictions, voting just returns the same answer three times - no improvement. The value comes from models that disagree on *different* samples: where LogReg gets it wrong, RF might get it right, and vice versa.

We can measure this directly by looking at how correlated the models' predictions are.

In [ ]:
# Fit models and get predictions to check correlation
from sklearn.model_selection import cross_val_predict

lr_preds = cross_val_predict(log_clf, X, y, cv=5)
rf_preds = cross_val_predict(rf_clf, X, y, cv=5)
svc_preds = cross_val_predict(svc_clf, X, y, cv=5)

pred_df = pd.DataFrame({
    "LogReg": lr_preds,
    "RF": rf_preds,
    "SVC": svc_preds,
})

corr = pred_df.corr()

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=0.5, vmax=1.0, ax=ax, fmt=".3f")
ax.set_title("Prediction Correlation Between Models")
plt.tight_layout()
plt.show()

Lower correlation means more diversity - and more potential for the ensemble to improve. If you see correlations close to 1.0 everywhere, voting won't help much. The ideal ensemble members are individually accurate *and* make different mistakes.

We can see this directly. Where in the feature space does each model get things right that the other gets wrong?

In [ ]:
# Categorize each sample by RF/SVC agreement
rf_correct = rf_preds == y
svc_correct = svc_preds == y

categories = np.full(len(y), "Both right", dtype=object)
categories[rf_correct & ~svc_correct] = "RF right, SVC wrong"
categories[~rf_correct & svc_correct] = "SVC right, RF wrong"
categories[~rf_correct & ~svc_correct] = "Both wrong"

# Show only where models disagree - these are the samples where ensembles can help
n_agree = (categories == "Both right").sum()
n_both_wrong = (categories == "Both wrong").sum()

fig, ax = plt.subplots(figsize=(8, 5))
for cat, color in [("RF right, SVC wrong", "steelblue"), ("SVC right, RF wrong", "tomato")]:
    mask = categories == cat
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, s=30, alpha=0.6,
               label=f"{cat} ({mask.sum()})", edgecolors="none")

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Where RF and SVC Disagree")
ax.legend(fontsize=9, markerscale=2)
plt.tight_layout()
plt.show()

print(f"Both models agree and are correct: {n_agree}")
print(f"Both models are wrong:             {n_both_wrong}")

The blue and red points are where the models disagree - and where an ensemble can win. If RF is right where SVC is wrong (blue) and SVC is right where RF is wrong (red), combining them captures both advantages. The remaining samples (printed below the plot) are where both models agree - an ensemble can't change those outcomes, for better or worse.

`VotingRegressor` works the same way for regression, averaging predicted values instead of probabilities.

---

## Part 3: Stacking Ensembles

### Learning how to combine

Voting treats all models equally (or with fixed weights). Stacking goes further: it trains a *meta-learner* that looks at the base models' predictions and learns the optimal way to combine them. The meta-learner can give more weight to models that are reliable in certain regions of the feature space and less weight to models that are noisy there.

The architecture:

1. Base models (RF, GBM, SVC, etc.) each generate predictions
2. Those predictions become *features* for a meta-learner (typically LogisticRegression)
3. The meta-learner learns how to combine the base predictions into a final answer

The critical detail: the predictions fed to the meta-learner must be *out-of-fold*. If the base models predict on data they trained on, the meta-learner sees artificially good inputs and overfits. sklearn's `StackingClassifier` handles this automatically - it uses cross-validation internally to generate honest predictions for the meta-learner.

In [ ]:
# Base models for stacking
base_models = [
    ("lr", Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(random_state=42))])),
    ("rf", RandomForestClassifier(n_estimators=200, random_state=42)),
    ("gb", GradientBoostingClassifier(n_estimators=200, random_state=42)),
    ("svc", Pipeline([("scaler", StandardScaler()), ("svc", SVC(probability=True, random_state=42))])),
]

# Meta-learner: LogisticRegression - simple, interpretable, avoids overfitting
stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(random_state=42),
    cv=5,
    stack_method="predict_proba",  # feed probabilities, not hard predictions
)

cv_stacking = cross_val_score(stacking_clf, X, y, cv=5, scoring="accuracy")
print(f"Stacking CV Accuracy: {cv_stacking.mean():.4f} ± {cv_stacking.std():.4f}")

### How the meta-learner sees the data

It's worth understanding what the meta-learner actually receives as input. Instead of the original 20 features, it gets the predicted probabilities from each base model. With `stack_method="predict_proba"` and binary classification, each model contributes two probabilities (class 0 and class 1), so the meta-learner fits on 8 features from 4 base models. That's still a much simpler problem than the original 20-feature dataset, which is why LogisticRegression works well as the meta-learner.

In [ ]:
# Demonstrate what the meta-learner sees
# Fit stacking on full data to inspect
stacking_clf.fit(X, y)

# With predict_proba and binary classes, each model contributes 2 coefficients
# (one per class probability). Sum the absolute values to get net influence.
coefs = stacking_clf.final_estimator_.coef_[0]
model_names = [name for name, _ in base_models]
# Each model contributes 2 consecutive columns (class 0 prob, class 1 prob)
net_influence = [
    np.sum(np.abs(coefs[i*2:(i+1)*2]))
    for i in range(len(model_names))
]

fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(model_names, net_influence, color="steelblue", edgecolor="black")
ax.set_xlabel("Net Influence (sum of absolute coefficients)")
ax.set_title("How the Meta-Learner Weighs Base Models")
plt.tight_layout()
plt.show()

The meta-learner gives the most weight to LogReg - surprising since it's the weakest individual model. But the meta-learner isn't choosing the best model; it's learning what each model's predictions *add* to the mix. LogReg's predictions are the least correlated with the others (see the heatmap above), so they carry the most unique information. SVC and GB, being more correlated with RF, add less that RF doesn't already provide.

This is exactly why stacking can include a weak model and still benefit from it, while voting cannot. Voting averages predictions equally, so a weak model drags the score down. Stacking *learns* how to use each model's signal, giving weak-but-different models a role that strong-but-redundant models can't fill.

`StackingRegressor` works identically for regression - base models predict values, meta-learner combines them.

### When stacking helps

Stacking adds real complexity: it's slower to train (every base model runs CV internally), harder to interpret, and can overfit if the base models are too similar. It works best when:

- Base models are genuinely diverse (linear + tree + distance-based)
- The dataset has enough samples for the internal CV to produce stable estimates
- Individual models are reasonably accurate but disagree on different subsets

On small or clean datasets, the improvement over voting (or even a single well-tuned model) may be negligible. The comparison table at the end will show whether it was worth it here.

---

## Part 4: Nested Cross-Validation

### The optimism problem

Throughout this course, we've used `cross_val_score` to estimate model performance and `GridSearchCV` / `RandomizedSearchCV` to tune hyperparameters. But there's a subtle issue when you combine the two.

When `RandomizedSearchCV` searches over hyperparameters, it uses cross-validation internally to pick the best configuration. If you then report that best CV score as your performance estimate, you've used the same data for both selecting and evaluating the model. The reported score is optimistically biased - it looks better than what you'd get on truly new data.

This is the same kind of data leakage we warned about with train/test splits in the very first lecture. The solution is the same principle: keep evaluation data completely separate from selection data.

### Nested CV: two loops

Nested cross-validation uses two CV loops:

- *Inner loop*: tunes hyperparameters (this is your `RandomizedSearchCV`)
- *Outer loop*: evaluates the entire tuning process on data the inner loop never saw

The outer loop tells you: "if I follow this tuning procedure on new data, how well will the resulting model perform?" This is an honest estimate of the *process*, not just one particular model.

Here's the structure visually. Each row is one outer fold. The inner loop runs entirely within the training portion - the outer test fold is never touched until final evaluation.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

n_outer = 5
n_inner = 4  # inner folds within each outer training set
colors = {"outer_test": "tomato", "inner_test": "gold", "inner_train": "steelblue"}

for outer_fold in range(n_outer):
    y_pos = n_outer - outer_fold - 1

    for block in range(n_outer):
        x_start = block * 1.1
        if block == outer_fold:
            # Outer test fold
            ax.barh(y_pos, 1.0, left=x_start, height=0.6, color=colors["outer_test"],
                    edgecolor="black", linewidth=0.5)
            ax.text(x_start + 0.5, y_pos, "Test\n(outer)", ha="center", va="center", fontsize=7)
        else:
            # Inner folds within the training portion
            inner_width = 1.0 / n_inner
            for j in range(n_inner):
                c = colors["inner_test"] if j == (outer_fold % n_inner) else colors["inner_train"]
                ax.barh(y_pos, inner_width * 0.95, left=x_start + j * inner_width,
                        height=0.6, color=c, edgecolor="gray", linewidth=0.3)

ax.set_yticks(range(n_outer))
ax.set_yticklabels([f"Outer fold {i+1}" for i in range(n_outer - 1, -1, -1)])
ax.set_xlim(-0.2, n_outer * 1.1 + 0.2)
ax.set_xticks([])
ax.set_title("Nested Cross-Validation Structure")

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=colors["inner_train"], edgecolor="gray", label="Inner train (tune hyperparams)"),
    Patch(facecolor=colors["inner_test"], edgecolor="gray", label="Inner validation (select best params)"),
    Patch(facecolor=colors["outer_test"], edgecolor="black", label="Outer test (evaluate process)"),
]
ax.legend(handles=legend_elements, loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

The outer test fold (red) is held completely aside - the inner loop never sees it. The inner loop splits the remaining data into train (blue) and validation (gold) to tune hyperparameters. Once the best hyperparameters are found, the outer test fold scores the result. This guarantees that the evaluation is honest.

In [ ]:
from xgboost import XGBClassifier

# Define the tuning process (inner loop)
param_distributions = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample": [0.8, 0.9, 1.0],
}

inner_cv = RandomizedSearchCV(
    XGBClassifier(random_state=42),
    param_distributions=param_distributions,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
)

# "Naive" score: the best score from RandomizedSearchCV
inner_cv.fit(X, y)
naive_score = inner_cv.best_score_
print(f"Naive CV score (from RandomizedSearchCV): {naive_score:.4f}")

# Nested CV score: wrap the entire search in an outer CV loop
nested_scores = cross_val_score(inner_cv, X, y, cv=5, scoring="accuracy")
print(f"Nested CV score:                          {nested_scores.mean():.4f} ± {nested_scores.std():.4f}")
print(f"Optimism gap:                             {naive_score - nested_scores.mean():.4f}")

The gap between the naive score and the nested score is the optimism - how much the naive estimate overpromises. On this dataset the gap may be small (a few tenths of a percent) or larger depending on how aggressive the search is. Either way, the nested score is the one you should report.

Note how simple the code is: `cross_val_score` with a `RandomizedSearchCV` object as the estimator. sklearn handles the nesting automatically - the inner search runs independently inside each outer fold.

### When to use nested CV

Nested CV is computationally expensive. With 5 outer folds and 20 inner iterations of 5-fold CV, you're fitting $5 \times 20 \times 5 = 500$ models. Use it when:

- You're reporting a final performance estimate (paper, presentation, project report)
- You've done extensive hyperparameter tuning and want an honest assessment
- You're comparing a heavily tuned model against a simpler one and need a fair comparison

For exploratory work and model development, regular CV with a held-out test set is fine. Nested CV is for the final score you stand behind.

---

## Part 5: Full Comparison

Let's bring everything together. How do the individual models, voting, and stacking compare on this dataset?

In [ ]:
all_models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "LogReg": Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(random_state=42))]),
    "KNN": Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier())]),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, random_state=42),
    "SVC": Pipeline([("scaler", StandardScaler()), ("svc", SVC(random_state=42))]),
    "Voting (soft)": voting_clf,
    "Stacking": stacking_clf,
}

comparison_rows = []
for name, model in all_models.items():
    start = time.time()
    cv = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    elapsed = time.time() - start
    comparison_rows.append({
        "Model": name,
        "CV Accuracy": f"{cv.mean():.4f}",
        "Std": f"{cv.std():.4f}",
        "Time (s)": f"{elapsed:.1f}",
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

In [ ]:
# Visualize
acc_vals = [float(r["CV Accuracy"]) for r in comparison_rows]
model_names = [r["Model"] for r in comparison_rows]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["gray"] * 7 + ["steelblue", "tomato"]  # highlight ensembles
bars = ax.barh(model_names, acc_vals, color=colors, edgecolor="black")
ax.set_xlabel("CV Accuracy")
ax.set_title("Individual Models vs. Combined Ensembles")
ax.invert_yaxis()

for bar, val in zip(bars, acc_vals):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=10)

plt.tight_layout()
plt.show()

The comparison tells the story. If stacking or voting beat the best individual model, combining was worth it. If the improvement is marginal, a single well-tuned model may be the better choice - simpler, faster, easier to explain. There's no universal answer; the data decides.

---

## Summary

| Strategy | How it works | Strengths | Weaknesses |
|---|---|---|---|
| Voting | Average predictions from diverse models | Simple, transparent, no extra training | Fixed combination, can't adapt |
| Stacking | Meta-learner trained on base model predictions | Learns optimal combination, flexible | Slower, more complex, overfitting risk |
| Nested CV | Outer CV evaluates entire tuning process | Honest performance estimate | Computationally expensive |

When to reach for each:

- *Voting*: quick improvement when you already have several trained models. Low overhead, easy to explain.
- *Stacking*: when you need maximum accuracy and have diverse base models with enough data for the internal CV.
- *Nested CV*: when you need an honest final score to report. Not for everyday model development.

The progression across the unit:

| Lecture | Strategy | Combines | Key idea |
|---|---|---|---|
| 13a | Bagging (RF) | Same model, different data | Variance reduction |
| 13b | Boosting (GBM, XGBoost) | Same model, sequential | Bias reduction |
| 14a | Voting / Stacking | Different models | Diversity |